# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs with their Croissant `@id` values.

In [ ]:
# Collect information about record sets, their fields, and columns by @id
from pprint import pprint

if not dataset.record_sets:
    print('No record sets defined in the Croissant metadata.')
else:
    print('Available Record Sets and their Fields:')
    for rset in dataset.record_sets:
        print(f"- Record set name: {rset.name}")
        print(f"  @id: {rset.id}")
        print(f"  Description: {getattr(rset, 'description', '')}")

        if hasattr(rset, 'fields') and rset.fields:
            print("  Fields:")
            for field in rset.fields:
                print(f"    - {field.name} (@id: {field.id}, dataType: {getattr(field, 'dataType', None)}, column: {getattr(field, 'column', None)})")
        else:
            print("    [No fields defined]")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
You must use the record set and field `@id`s from the overview.

In [ ]:
# Get all record set @id's

# Note: For the FAIR2 dataset, the main record set typically describes the main data table.
record_set_ids = [rset.id for rset in dataset.record_sets]
dataframes = {}

for rset_id in record_set_ids:
    # Load all records as a DataFrame
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded records for record set {rset_id}: {df.shape[0]} rows, {df.shape[1]} columns.")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print('-'*60)

# Use the first available record set for EDA as an example
main_rset_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we:
- Select a numeric field by its `@id` (column name)
- Filter on a threshold
- Normalize the selected field
- Group by a categorical field if available

All references to fields or columns must be by their `@id`.

In [ ]:
if main_rset_id and main_rset_id in dataframes:
    df = dataframes[main_rset_id].copy()
    print(f"Working EDA on Record Set: {main_rset_id}")
    # Try to auto-detect a numeric field (@id or column name)
    potential_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not potential_numeric_fields:
        # Try to convert columns to numeric, skip errors
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        potential_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if not potential_numeric_fields:
        print("No numeric fields found for EDA.")
    else:
        numeric_field_id = potential_numeric_fields[0]  # first numeric field by @id
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'iufc' else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (Threshold=@mean): {len(filtered_df)} records")
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by the first available categorical field (@id)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object and df[col].nunique() < 20:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (showing mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No main record set or data available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. All field and column references must be by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rset_id and main_rset_id in dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_rset_id][numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping is available
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_rset_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('Not enough information to build plots.')

## 6. Conclusion

- We have loaded the FAIR² mass spectrometry dataset using the `mlcroissant` library.
- Explored the record sets and their columns by Croissant `@id`.
- Loaded records into Pandas DataFrames for further analysis.
- Performed basic EDA including filtering, normalization, and grouping by categorical variables.
- Made basic visualizations of one of the numeric fields.

**Next steps:** Deeper domain-specific analysis, machine learning, or statistical hypothesis testing, using the clean data and clearly documented field provenance via Croissant metadata.